In [19]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import math

In [20]:
def local2llh(x_km, y_km, origin_lon_deg, origin_lat_deg):
    """
    Inverse polyconic projection: local XY (km) -> lon/lat (deg), WGS84.

    Parameters
    ----------
    x_km : array_like
        Local x coordinates in km (positive east).
    y_km : array_like
        Local y coordinates in km (positive north).
    origin_lon_deg : float
        Origin longitude in degrees.
    origin_lat_deg : float
        Origin latitude in degrees.

    Returns
    -------
    lon_deg : ndarray
        Longitudes in degrees.
    lat_deg : ndarray
        Latitudes in degrees.
    h : ndarray
        Heights (always 0 here; vertical not used).
    
    example:
    lon, lat, h = local2llh(x_km, y_km, origin_lon, origin_lat)
    """
    # ---- constants (WGS84) ----
    a = 6378137.0
    e = 0.08209443794970

    # ---- convert inputs ----
    x = np.asarray(x_km, dtype=float) * 1000.0  # km -> m
    y = np.asarray(y_km, dtype=float) * 1000.0

    lon0 = math.radians(origin_lon_deg)
    lat0 = math.radians(origin_lat_deg)
    origin = np.array([lon0, lat0], dtype=float)

    # meridional arc function M(phi)
    def Mfunc(phi):
        return a * (
            (1 - e**2/4 - 3*e**4/64 - 5*e**6/256) * phi
            - (3*e**2/8 + 3*e**4/32 + 45*e**6/1024) * np.sin(2 * phi)
            + (15*e**4/256 + 45*e**6/1024) * np.sin(4 * phi)
            - (35*e**6/3072) * np.sin(6 * phi)
        )

    # ---- initial terms ----
    M0 = Mfunc(origin[1])

    # MATLAB: z = xy(2,:) ~= -M0;
    # here y != -M0 (mask for "normal" case)
    z = np.abs(y + M0) > 1e-9

    A = (M0 + y[z]) / a
    B = x[z]**2 / a**2 + A**2

    # allocate lon/lat arrays (radians)
    lat = np.zeros_like(x, dtype=float)
    lon = np.zeros_like(x, dtype=float)

    # initial guess for latitude where z is True
    lat[z] = A.copy()

    delta = np.full_like(A, np.inf, dtype=float)
    c = 0

    # ---- iterative solution for latitude ----
    while np.max(np.abs(delta)) > 1e-8:
        C = np.sqrt(1 - e**2 * np.sin(lat[z])**2) * np.tan(lat[z])
        M = Mfunc(lat[z])

        Mn = (
            1 - e**2/4 - 3*e**4/64 - 5*e**6/256
            - 2 * (3*e**2/8 + 3*e**4/32 + 45*e**6/1024) * np.cos(2 * lat[z])
            + 4 * (15*e**4/256 + 45*e**6/1024) * np.cos(4 * lat[z])
            - 6 * (35*e**6/3072) * np.cos(6 * lat[z])
        )

        Ma = M / a

        num = -(A * (C * Ma + 1) - Ma - 0.5 * (Ma**2 + B) * C)
        den = (
            e**2 * np.sin(2 * lat[z]) * (Ma**2 + B - 2 * A * Ma) / (4 * C)
            + (A - Ma) * (C * Mn - 2 / np.sin(2 * lat[z]))
            - Mn
        )

        delta = num / den
        lat[z] = lat[z] + delta

        c += 1
        if c > 100:
            raise RuntimeError("Convergence failure.")

    # ---- compute longitude for normal case (z) ----
    C = np.sqrt(1 - e**2 * np.sin(lat[z])**2) * np.tan(lat[z])
    lon[z] = np.arcsin(x[z] * C / a) / np.sin(lat[z]) + lon0

    # ---- special case: latitude = 0 ----
    lon[~z] = x[~z] / a + lon0
    lat[~z] = 0.0

    # ---- back to degrees ----
    lon_deg = np.degrees(lon)
    lat_deg = np.degrees(lat)
    h = np.zeros_like(lon_deg)

    return lon_deg, lat_deg, h



def llh2local(lon_deg, lat_deg, origin_lon_deg, origin_lat_deg):
    """
    Convert longitude/latitude (deg) to local Cartesian coordinates (km)
    using a polyconic projection, WGS84 ellipsoid.
    

    Parameters
    ----------
    lon_deg : array_like
        Longitudes in degrees (can be scalar or array).
    lat_deg : array_like
        Latitudes in degrees (same shape as lon_deg).
    origin_lon_deg : float
        Origin longitude in degrees.
    origin_lat_deg : float
        Origin latitude in degrees.

    Returns
    -------
    x_km : ndarray
        Local x coordinates in km (positive east).
    y_km : ndarray
        Local y coordinates in km (positive north).
    example:
    x, y = llh2local(lon, lat, origin_lon, origin_lat)
    """
    # WGS84
    a = 6378137.0
    e = 0.08209443794970

    lon = np.asarray(lon_deg, dtype=float)
    lat = np.asarray(lat_deg, dtype=float)

    lon0 = math.radians(origin_lon_deg)
    lat0 = math.radians(origin_lat_deg)

    phi = np.radians(lat)
    lam = np.radians(lon)

    # mask for latitude != 0
    z = phi != 0.0

    x = np.zeros_like(phi, dtype=float)
    y = np.zeros_like(phi, dtype=float)

    dlambda = lam - lon0

    def Mfunc(phi_val):
        return a * (
            (1 - e**2/4 - 3*e**4/64 - 5*e**6/256) * phi_val
            - (3*e**2/8 + 3*e**4/32 + 45*e**6/1024) * np.sin(2*phi_val)
            + (15*e**4/256 + 45*e**6/1024) * np.sin(4*phi_val)
            - (35*e**6/3072) * np.sin(6*phi_val)
        )

    M  = Mfunc(phi[z])
    M0 = Mfunc(lat0)

    N  = a / np.sqrt(1 - e**2 * np.sin(phi[z])**2)
    E  = dlambda[z] * np.sin(phi[z])

    # xy(1,z) = Easting, xy(2,z) = Northing in MATLAB
    x[z] = N * np.cos(phi[z]) / np.sin(phi[z]) * np.sin(E)
    y[z] = M - M0 + N * np.cos(phi[z]) / np.sin(phi[z]) * (1 - np.cos(E))

    # Special case: latitude == 0
    dlambda0 = dlambda[~z]
    x[~z] = a * dlambda0
    y[~z] = -M0

    # Convert to km
    x_km = x / 1000.0
    y_km = y / 1000.0

    return x_km, y_km


In [29]:
# read data
new_ductile = pd.read_csv("ductile_points_09122025.csv")

# x = lon, y = lat (in degrees)
lon = new_ductile["x"].values
lat = new_ductile["y"].values

# origin in degrees
origin_lon = 37.019
origin_lat = 37.23

# convert to local coordinates (km)
x_local_km, y_local_km = llh2local(lon, lat, origin_lon, origin_lat)

# add to dataframe (or overwrite)
new_ductile["dgammadot0"] = 0
new_ductile["x_local_m"] = x_local_km * 1000
new_ductile["y_local_m"] = y_local_km * 1000
new_ductile["z_local_m"] = 30000
new_ductile["length"] = 700000
new_ductile["width"] = 10000
new_ductile["thickness"] = 18000
new_ductile["strike"] = 90
new_ductile["dip"] = 90


# save file
outfile = "weak_zones.txt"

# number of weak zones
N = len(new_ductile)

with open(outfile, "w") as f:
    # header lines
    f.write("# number of linear weak zones\n")
    f.write(f"{N}\n")
    f.write("#nb\tdgammadot0\tx1\tx2\tx3\tlength\twidth\tthickness\tstrike\tdip\n")

    # write each row
    for i, row in new_ductile.iterrows():
        nb = i + 1  # numbering starting at 1

        f.write(
            f"{nb:4d}\t"
            f"{row['dgammadot0']:14.10f}\t"
            f"{int(row['y_local_m']):10d}\t"
            f"{int(row['x_local_m']):10d}\t"
            f"{int(row['z_local_m']):10d}\t"
            f"{int(row['length']):10d}\t"
            f"{int(row['width']):10d}\t"
            f"{int(row['thickness']):10d}\t"
            f"{int(row['strike']):5d}\t"
            f"{int(row['dip']):5d}\n"
        )


In [26]:
new_ductile

,id,distance,angle,x,y,x_local_m,y_local_m,z_local_m,length,width,thickness,strike,dip
0,NaN,0.1,41.552613,35.707178,35.358516,-119226.613026,-206873.971717,30000,70000,10000,18000,90,90
1,NaN,0.3,41.552613,35.839839,35.508185,-106971.970494,-190419.258787,30000,70000,10000,18000,90,90
2,NaN,0.5,41.552613,35.972501,35.657854,-94761.074823,-173948.483386,30000,70000,10000,18000,90,90
3,NaN,0.7,21.370622,36.094701,35.813925,-83532.958824,-156742.435801,30000,70000,10000,18000,90,90
4,NaN,0.9,21.370622,36.167581,36.000173,-76766.555297,-136136.264174,30000,70000,10000,18000,90,90
5,NaN,1.1,21.370622,36.240461,36.186422,-70030.200533,-115524.790263,30000,70000,10000,18000,90,90
6,NaN,1.3,21.370622,36.313340,36.372670,-63324.107753,-94907.974767,30000,70000,10000,18000,90,90
7,NaN,1.5,21.370622,36.386220,36.558919,-56648.489648,-74285.779606,30000,70000,10000,18000,90,90
8,NaN,1.7,358.215157,36.454852,36.745901,-50382.443899,-53574.577844,30000,70000,10000,18000,90,90
9,NaN,1.9,358.215157,36.448623,36.945804,-50806.325522,-31387.254440,30000,70000,10000,18000,90,90
